In [97]:
import calendar
import time
from DATA.stock_invest_function import *
from datetime import datetime
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore')

In [98]:
# 유틸리티 함수들
def convert_to_month_end(date_str):
    try:
        # 문자열/타입 혼용 안전 변환
        date_obj = pd.to_datetime(date_str)
        if pd.isna(date_obj):
            return None

        y, m, d = date_obj.year, date_obj.month, date_obj.day

        # 1~5일 → 전달 말일
        if 1 <= d <= 5:
            if m == 1:
                prev_y, prev_m = y - 1, 12
            else:
                prev_y, prev_m = y, m - 1
            last_day_prev = calendar.monthrange(prev_y, prev_m)[1]
            return datetime(prev_y, prev_m, last_day_prev)

        # 그 외 → 해당월 말일
        last_day_cur = calendar.monthrange(y, m)[1]
        return datetime(y, m, last_day_cur)

    except Exception:
        return None

def process_daily_to_monthly_market_data(daily_data, ticker):
    if not daily_data:
        return pd.DataFrame()
    df = pd.DataFrame(daily_data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df['year_month'] = df['date'].dt.to_period('M')
    monthly_data = []
    for year_month in df['year_month'].unique():
        month_data = df[df['year_month'] == year_month]
        last_day_data = month_data.loc[month_data['date'].idxmax()]
        monthly_data.append({
            'ticker': ticker,
            'date': last_day_data['date'],
            'market_cap': last_day_data['marketCap'],
            'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
        })
    return pd.DataFrame(monthly_data)


def fetch_revenue_data(ticker, api_key):
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {'limit': 200, 'apikey': api_key, 'period': 'quarter'}
    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code != 200:
            return None, f"HTTP {response.status_code}"
        data = response.json()
        if isinstance(data, dict) and 'Error Message' in data:
            return None, f"API 오류: {data['Error Message']}"
        if not data:
            return None, "데이터 없음"
        return data, None
    except Exception as e:
        return None, f"오류: {str(e)}"

def fetch_market_data_yearly(ticker, api_key, start_year=2010):
    all_data = []
    current_year = datetime.now().year
    for year in range(start_year, current_year + 1):
        start_date_str = f"{year}-01-01"
        end_date_str = f"{year}-12-31"
        url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
        params = {'from': start_date_str, 'to': end_date_str, 'apikey': api_key}
        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if data and isinstance(data, list):
                    all_data.extend(data)
            time.sleep(0.3)
        except Exception as e:
            continue
    return all_data if all_data else None, None

def fetch_db_revenue_data(ticker, db_info, end_date='2025-08-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, saleq
        FROM US_fundq
        WHERE ticker = '{ticker}'
        AND saleq IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['revenue_billions'] = df['saleq'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'revenue_billions']]
    except Exception as e:
        return pd.DataFrame()

def fetch_db_market_data(ticker, db_info, end_date='2024-12-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, me
        FROM US_fundm
        WHERE ticker = '{ticker}'
        AND me IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['market_cap_billions'] = df['me'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'market_cap_billions']]
    except Exception as e:
        return pd.DataFrame()

def calculate_enhanced_ttm_and_psr(merged_data):
    """Calculate enhanced TTM and PSR"""
    df = merged_data.copy()
    df = df.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

    # Calculate TTM from quarterly revenue
    df['revenue_ttm'] = df.groupby('ticker')['revenue_billions'].rolling(window=4, min_periods=1).sum().reset_index(0,
                                                                                                                    drop=True)
    df['revenue_ttm_billions'] = df['revenue_ttm']

    # Apply 2-month shift
    df['revenue_ttm_shift'] = df.groupby('ticker')['revenue_ttm_billions'].shift(2)

    # Calculate PSR
    df['PSR_ttm'] = df['market_cap_billions'] / df['revenue_ttm_shift']

    # Handle infinite values
    df['PSR_ttm'] = df['PSR_ttm'].replace([np.inf, -np.inf], np.nan)

    return df

In [99]:
# 설정값들
ticker = 'AAPL'

hs_code = '841191'

api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

start_date_month = '2011-03-01'
end_date_month = (pd.Timestamp.today().normalize() - pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')

In [100]:
print(end_date_month)

2025-08-31


In [101]:
print("=" * 80)
print("전처리 과정 테스트 시작")
print(f"대상 종목: {ticker}")
print("=" * 80)

# 1. FMP 매출 데이터 수집
print("\n1. FMP 매출 데이터 수집 중...")
revenue_data, error = fetch_revenue_data(ticker, api_key)

if revenue_data is None:
    print(f"ERROR: FMP 매출 데이터 수집 실패 - {error}")
    exit()

all_revenue_data = []
for item in revenue_data:
    all_revenue_data.append({
        'ticker': ticker,
        'date': item.get('date', ''),
        'calendar_year': item.get('calendarYear', ''),
        'period': item.get('period', ''),
        'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
        'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
    })

fmp_revenue_df = pd.DataFrame(all_revenue_data)
fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])
fmp_revenue_df['date_month_end'] = fmp_revenue_df['date'].apply(convert_to_month_end)
fmp_revenue_df = fmp_revenue_df.drop_duplicates(subset=['date_month_end'], keep='first').reset_index(drop=True)
print(f"FMP 매출 데이터: {len(fmp_revenue_df)}건")


# 2. DB 매출 데이터 가져오기
db_revenue_raw = fetch_db_revenue_data(ticker, db_info)
db_revenue_df = db_revenue_raw.loc[db_revenue_raw['revenue_billions'] != db_revenue_raw['revenue_billions'].shift()]

# db_revenue_df = db_revenue_raw.drop_duplicates(subset=['date_month_end', 'revenue_billions'], keep='first')
mereged_rev_data = pd.merge(fmp_revenue_df, db_revenue_df, on = ['ticker', 'date_month_end'], how='outer')

rev_data = mereged_rev_data[mereged_rev_data['date_month_end'] >= data_start_date]
rev_data['revenue_billions_x'] = rev_data['revenue_billions_x'].fillna(rev_data['revenue_billions_y'])
# 컬럼 이름 변경
rev_data.rename(columns={'revenue_billions_x': 'revenue_billions'}, inplace=True)


전처리 과정 테스트 시작
대상 종목: AAPL

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 160건


In [104]:
import importlib
import DATA.us_sarima_forecast as sarima
importlib.reload(sarima)
import DATA.us_lstm_forecast_v2 as lstm_v2
importlib.reload(lstm_v2)
import DATA.us_prophet_forecast_v3 as prophet_v3
importlib.reload(prophet_v3)
import DATA.us_est_forecast_v2 as esmod
importlib.reload(esmod)

<module 'DATA.us_est_forecast_v2' from 'C:\\Users\\MetaM\\PycharmProjects\\stock_forecast\\DATA\\us_est_forecast_v2.py'>

In [105]:
# periods=4 또는 8 등 원하는 분기 수
periods = 4

# 모듈 함수 호출 (내부에서 월말 정렬/중복제거 처리)
sarima_df, results = sarima.run_sarima_prediction(
    rev_data,
    forecast_quarters=periods,   # ← 예측 분기 수
    exog_col=None                # 외생변수 없으면 None
)

# 인덱스를 date_month_end로 설정
sarima_df = sarima_df.sort_values("date_month_end").set_index("date_month_end")
print(sarima_df.tail(12))

TypeError: cannot unpack non-iterable NoneType object

In [84]:
# 1) 4분기 예측
# 4분기 예측
lstm_raw_df, lstm_results_4q = lstm_v2.run_lstm_revenue_prediction(rev_data, ticker=ticker, prediction_quarters=4)
lstm_df = lstm_raw_df.drop_duplicates(subset=['revenue_billions_lstm_forecast'], keep='last')

In [85]:
# 4분기 예측
prophet_raw_df, res_4q = prophet_v3.run_prophet_revenue_only(rev_data, ticker=ticker, prediction_quarters=4)

17:09:29 - cmdstanpy - INFO - Chain [1] start processing
17:09:30 - cmdstanpy - INFO - Chain [1] done processing


In [86]:
# 4분기 예측
es_raw_df, res_q4 = esmod.run_es_revenue_quarterly(rev_data, ticker=ticker, prediction_quarters=4)
# es_raw_df.tail(24)


In [87]:
# 3. FMP 시가총액 데이터 수집
print("2. FMP 시가총액 데이터 수집 중...")
market_data, error = fetch_market_data_yearly(ticker, api_key, start_year=2010)

if not market_data:
    print("ERROR: FMP 시가총액 데이터 수집 실패")
    raise SystemExit(1)

fmp_market_df = process_daily_to_monthly_market_data(market_data, ticker).copy()
fmp_market_df['date_month_end'] = fmp_market_df['date'].apply(convert_to_month_end)
# 혹시 중복/정렬 문제 예방
fmp_market_df = (fmp_market_df
                 .drop_duplicates(subset=['date_month_end'])
                 .sort_values('date_month_end')
                 .reset_index(drop=True))

print(f"FMP 시가총액 데이터: {len(fmp_market_df)}건")

# -----------------------------
# 안전 병합: DB가 없으면 FMP만 사용
# -----------------------------
def _safe_get_db_market_df():
    try:
        df = fetch_db_market_data(ticker, db_info)
        # None 이거나 길이 0이면 빈 DF 반환
        if df is None or len(df) == 0:
            return pd.DataFrame()
        return df.copy()
    except Exception as e:
        print(f"[WARN] DB 조회 중 예외 발생: {e}")
        return pd.DataFrame()

db_market_df = _safe_get_db_market_df()

# DB가 있으면 date_month_end 정규화 + 컬럼 정리
if not db_market_df.empty:
    # 날짜 컬럼 유도: date_month_end가 없고 date가 있으면 생성
    if 'date_month_end' not in db_market_df.columns:
        if 'date' in db_market_df.columns:
            db_market_df['date_month_end'] = db_market_df['date'].apply(convert_to_month_end)
        else:
            # 날짜 정보가 없으면 병합 불가 → 빈 DF 취급
            print("[WARN] DB 데이터에 날짜 컬럼이 없어 병합을 건너뜁니다.")
            db_market_df = pd.DataFrame()

if db_market_df.empty:
    # DB가 비어 있으면 FMP만 사용
    print("[INFO] DB 시가총액 데이터 없음 → FMP 데이터만 사용합니다.")
    merged_market_df = fmp_market_df.copy()
    # from_db 컬럼은 NaN으로 생성(분석 시 출처 구분 유용)
    merged_market_df['market_cap_billions_from_db'] = np.nan

else:
    # 필요한 컬럼명 정리
    # DB에 market_cap_billions가 있으면 rename, 없으면 NaN으로 준비
    if 'market_cap_billions' in db_market_df.columns:
        db_market_df_renamed = db_market_df.rename(
            columns={'market_cap_billions': 'market_cap_billions_from_db'}
        )
    else:
        # 필요한 최소 컬럼만 추려서 NaN 채우기
        db_market_df_renamed = db_market_df[['date_month_end']].copy()
        db_market_df_renamed['market_cap_billions_from_db'] = np.nan
        print("[WARN] DB에 'market_cap_billions' 컬럼이 없어 NaN으로 채웁니다.")

    # 병합 (분기/월말 정렬 맞춤)
    merged_market_df = fmp_market_df.merge(
        db_market_df_renamed[['date_month_end', 'market_cap_billions_from_db']],
        on='date_month_end',
        how='left'   # FMP 기준으로 맞추고 DB 값 있으면 붙임
    )

# 최종 결측 보충: FMP 값이 NaN이면 DB 값으로 대체
if 'market_cap_billions' not in merged_market_df.columns:
    # 혹시 FMP 가 다른 이름을 썼다면 여기서 보정하세요.
    # 일단 없으면 새로 만들고 DB로 채움
    merged_market_df['market_cap_billions'] = np.nan

if 'market_cap_billions_from_db' not in merged_market_df.columns:
    merged_market_df['market_cap_billions_from_db'] = np.nan

merged_market_df['market_cap_billions'] = merged_market_df['market_cap_billions'].fillna(
    merged_market_df['market_cap_billions_from_db']
)

# 정리
merged_market_df = (merged_market_df
                    .drop_duplicates(subset=['date_month_end'])
                    .sort_values('date_month_end')
                    .reset_index(drop=True))

print(f"병합 완료: {len(merged_market_df)}건 (FMP+DB)")


2. FMP 시가총액 데이터 수집 중...
FMP 시가총액 데이터: 189건
병합 완료: 189건 (FMP+DB)


In [94]:
# merged_market_df

enhanced_merged_df = pd.merge(merged_market_df[['date_month_end', 'market_cap_billions']], rev_data, on='date_month_end', how='outer')

market_cap_resize = enhanced_merged_df[['date_month_end', 'market_cap_billions', 'ticker', 'revenue_billions']].copy()
market_cap_resize.dropna(subset =['market_cap_billions'], inplace=True)
market_cap_resize.ffill(limit=2, inplace=True)

market_cap_resize = market_cap_resize[(market_cap_resize['date_month_end'] >= start_date_month ) & (market_cap_resize['date_month_end'] <= end_date_month)]


In [95]:
# market_cap_resize
enhanced_merged_df_with_ttm = calculate_enhanced_ttm_and_psr(market_cap_resize)

In [96]:
enhanced_merged_df_with_ttm

,date_month_end,market_cap_billions,ticker,revenue_billions,revenue_ttm,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm
0,2011-03-31,322.84,AAPL,24.67,24.67,24.67,NaN,NaN
1,2011-04-30,324.14,AAPL,24.67,49.34,49.34,NaN,NaN
2,2011-05-31,322.06,AAPL,24.67,74.01,74.01,24.67,13.054722
3,2011-06-30,311.64,AAPL,28.57,102.58,102.58,49.34,6.316173
4,2011-07-31,362.59,AAPL,28.57,106.48,106.48,74.01,4.899203
...,...,...,...,...,...,...,...,...
169,2025-04-30,3166.86,AAPL,95.36,439.32,439.32,467.83,6.769254
170,2025-05-31,2993.24,AAPL,95.36,410.38,410.38,468.26,6.392261
171,2025-06-30,3057.63,AAPL,94.04,380.12,380.12,439.32,6.959915
172,2025-07-31,3093.39,AAPL,94.04,378.80,378.80,410.38,7.537867
